<a href="https://colab.research.google.com/github/hagarwaleed/Toxic_text-Classification/blob/main/Fine%20Tune%20DistilBERT%20with%20LoRA/Finetune_DistilBERT_LoRa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score , precision_score, recall_score
import torch
from transformers import TrainerCallback
from torch.utils.data import DataLoader, Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
df = pd.read_csv('cellula toxic data  (1).csv')

In [ ]:
del df['image descriptions']

# TEXT PREPROCESSING

In [ ]:
def preprocess_text_bert(text):
    """
    Basic preprocessing for BERT (preserves BERT's tokenization patterns)
    """

    text = str(text).lower()

    text = re.sub(r'\s+', ' ', text).strip()

    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)
    return text

In [ ]:
df['query_clean'] = df['query'].apply(preprocess_text_bert)

# LABEL ENCODING

In [ ]:
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['Toxic Category'])
num_classes = len(label_encoder.classes_)

# DATA SPLITTING

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label_encoded']
)

# TOKENIZER INITIALIZATION

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
max_length = 128

# CUSTOM DATASET CLASS

In [ ]:
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# DATASET CREATION

In [ ]:
train_dataset = ToxicDataset(train_df['query_clean'], train_df['label_encoded'], tokenizer, max_length)
test_dataset = ToxicDataset(test_df['query_clean'], test_df['label_encoded'], tokenizer, max_length)

# MODEL LOADING

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes,
    id2label={i: label for i, label in enumerate(label_encoder.classes_)},
    label2id={label: i for i, label in enumerate(label_encoder.classes_)}
)

# LoRA CONFIGURATION

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    # bias="none",  # Don't train bias parameters
)

# APPLYING LoRA

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# METRICS FUNCTION

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1_micro = f1_score(labels, predictions, average='micro')
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    recall = recall_score(labels, predictions, average='weighted')
    precision= precision_score(labels, predictions, average='weighted')

    return {
        "accuracy": accuracy,
        "recall": recall ,
        "precision": precision,
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

# TRAINING ARGUMENTS

In [ ]:
training_args = TrainingArguments(
    output_dir='./distilbert-lora-toxic',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=2,
)

# TRAINER SETUP

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
train_history = trainer.train()

# EVALUATION

In [ ]:
test_results = trainer.evaluate(eval_dataset=test_dataset)
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['label_encoded'].values

In [ ]:
# Calculate metrics
test_accuracy = accuracy_score(true_labels, pred_labels)
test_f1_micro = f1_score(true_labels, pred_labels, average='micro')
test_recall=recall_score(true_labels,pred_labels, average='weighted')
test_precision=precision_score(true_labels,pred_labels, average='weighted')

print(f"\n📊 TEST SET METRICS:")
print("-" * 40)
print(f"Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f} %)")
print(f"F1 Micro:  {test_f1_micro:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"Precision: {test_precision:.4f}")
print("-" * 40)
